# Chapter 3 - A Tour of Machine Learning Classifiers Using Scikit-Learn

### Overview

- [Choosing a classification algorithm](#Choosing-a-classification-algorithm)
- [First steps with scikit-learn](#First-steps-with-scikit-learn)
    - [Training a perceptron via scikit-learn](#Training-a-perceptron-via-scikit-learn)
- [Modeling class probabilities via logistic regression](#Modeling-class-probabilities-via-logistic-regression)
    - [Logistic regression intuition and conditional probabilities](#Logistic-regression-intuition-and-conditional-probabilities)
    - [Learning the weights of the logistic loss function](#Learning-the-weights-of-the-logistic-loss-function)
    - [Training a logistic regression model with scikit-learn](#Training-a-logistic-regression-model-with-scikit-learn)
    - [Tackling overfitting via regularization](#Tackling-overfitting-via-regularization)
- [Maximum margin classification with support vector machines](#Maximum-margin-classification-with-support-vector-machines)
    - [Maximum margin intuition](#Maximum-margin-intuition)
    - [Dealing with the nonlinearly separable case using slack variables](#Dealing-with-the-nonlinearly-separable-case-using-slack-variables)
    - [Alternative implementations in scikit-learn](#Alternative-implementations-in-scikit-learn)
- [Solving nonlinear problems using a kernel SVM](#Solving-nonlinear-problems-using-a-kernel-SVM)
    - [Using the kernel trick to find separating hyperplanes in higher dimensional space](#Using-the-kernel-trick-to-find-separating-hyperplanes-in-higher-dimensional-space)
- [Decision tree learning](#Decision-tree-learning)
    - [Maximizing information gain – getting the most bang for the buck](#Maximizing-information-gain-–-getting-the-most-bang-for-the-buck)
    - [Building a decision tree](#Building-a-decision-tree)
    - [Combining weak to strong learners via random forests](#Combining-weak-to-strong-learners-via-random-forests)
- [K-nearest neighbors – a lazy learning algorithm](#K-nearest-neighbors-–-a-lazy-learning-algorithm)
- [Summary](#Summary)

<br>
<br>

<br>
<br>

In [ ]:
from IPython.display import Image
%matplotlib inline

# Choosing a classification algorithm

...

# First steps with scikit-learn

Loading the Iris dataset from scikit-learn. Here, the third column represents the petal length, and the fourth column the petal width of the flower examples. The classes are already converted to integer labels where 0=Iris-Setosa, 1=Iris-Versicolor, 2=Iris-Virginica.

In [ ]:
from sklearn import datasets
import numpy as np

iris = datasets.load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

print('Class labels:', np.unique(y))

Splitting data into 70% training and 30% test data:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1, stratify=y)

In [ ]:
print('Labels counts in y:', np.bincount(y))
print('Labels counts in y_train:', np.bincount(y_train))
print('Labels counts in y_test:', np.bincount(y_test))

Standardizing the features:

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
sc.fit(X_train)
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

<br>
<br>

## Training a perceptron via scikit-learn

In [ ]:
from sklearn.linear_model import Perceptron

ppn = Perceptron(eta0=0.1, random_state=1)
ppn.fit(X_train_std, y_train)

In [ ]:
y_pred = ppn.predict(X_test_std)
print('Misclassified examples: %d' % (y_test != y_pred).sum())

In [ ]:
from sklearn.metrics import accuracy_score

print('Accuracy: %.3f' % accuracy_score(y_test, y_pred))

In [ ]:
print('Accuracy: %.3f' % ppn.score(X_test_std, y_test))

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

# To check recent matplotlib compatibility
import matplotlib
from distutils.version import LooseVersion


def plot_decision_regions(X, y, classifier, test_idx=None, resolution=0.02):

    # setup marker generator and color map
    markers = ('o', 's', '^', 'v', '<')
    colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
    cmap = ListedColormap(colors[:len(np.unique(y))])

    # plot the decision surface
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))
    lab = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
    lab = lab.reshape(xx1.shape)
    plt.contourf(xx1, xx2, lab, alpha=0.3, cmap=cmap)
    plt.xlim(xx1.min(), xx1.max())
    plt.ylim(xx2.min(), xx2.max())

    # plot class examples
    for idx, cl in enumerate(np.unique(y)):
        plt.scatter(x=X[y == cl, 0], 
                    y=X[y == cl, 1],
                    alpha=0.8, 
                    c=colors[idx],
                    marker=markers[idx], 
                    label=f'Class {cl}', 
                    edgecolor='black')

    # highlight test examples
    if test_idx:
        # plot all examples
        X_test, y_test = X[test_idx, :], y[test_idx]

        plt.scatter(X_test[:, 0],
                    X_test[:, 1],
                    c='none',
                    edgecolor='black',
                    alpha=1.0,
                    linewidth=1,
                    marker='o',
                    s=100, 
                    label='Test set')        

Training a perceptron model using the standardized training data:

In [ ]:
X_combined_std = np.vstack((X_train_std, X_test_std))
y_combined = np.hstack((y_train, y_test))

plot_decision_regions(X=X_combined_std, y=y_combined,
                      classifier=ppn, test_idx=range(105, 150))
plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')

plt.tight_layout()
#plt.savefig('figures/03_01.png', dpi=300)
plt.show()

<br>
<br>

# Modeling class probabilities via logistic regression

...

### Logistic regression intuition and conditional probabilities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.arange(-7, 7, 0.1)
sigma_z = sigmoid(z)

plt.plot(z, sigma_z)
plt.axvline(0.0, color='k')
plt.ylim(-0.1, 1.1)
plt.xlabel('z')
plt.ylabel('$\sigma (z)$')

# y axis ticks and gridline
plt.yticks([0.0, 0.5, 1.0])
ax = plt.gca()
ax.yaxis.grid(True)

plt.tight_layout()
#plt.savefig('figures/03_02.png', dpi=300)
plt.show()

In [ ]:
Image(filename='figures/03_03.png', width=500) 

In [ ]:
Image(filename='figures/03_25.png', width=500) 

<br>
<br>

### Learning the weights of the logistic loss function

In [ ]:
def loss_1(z):
    return - np.log(sigmoid(z))


def loss_0(z):
    return - np.log(1 - sigmoid(z))

z = np.arange(-10, 10, 0.1)
sigma_z = sigmoid(z)

c1 = [loss_1(x) for x in z]
plt.plot(sigma_z, c1, label='L(w, b) if y=1')

c0 = [loss_0(x) for x in z]
plt.plot(sigma_z, c0, linestyle='--', label='L(w, b) if y=0')

plt.ylim(0.0, 5.1)
plt.xlim([0, 1])
plt.xlabel('$\sigma(z)$')
plt.ylabel('L(w, b)')
plt.legend(loc='best')
plt.tight_layout()
#plt.savefig('figures/03_04.png', dpi=300)
plt.show()

In [ ]:
class LogisticRegressionGD:
    """Gradient descent-based logistic regression classifier.

    Parameters
    ------------
    eta : float
      Learning rate (between 0.0 and 1.0)
    n_iter : int
      Passes over the training dataset.
    random_state : int
      Random number generator seed for random weight
      initialization.


    Attributes
    -----------
    w_ : 1d-array
      Weights after training.
    b_ : Scalar
      Bias unit after fitting.
    losses_ : list
       Log loss function values in each epoch.

    """
    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X, y):
        """ Fit training data.

        Parameters
        ----------
        X : {array-like}, shape = [n_examples, n_features]
          Training vectors, where n_examples is the number of examples and
          n_features is the number of features.
        y : array-like, shape = [n_examples]
          Target values.

        Returns
        -------
        self : Instance of LogisticRegressionGD

        """
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = np.float_(0.)
        self.losses_ = []

        for i in range(self.n_iter):
            net_input = self.net_input(X)
            output = self.activation(net_input)
            errors = (y - output)
            self.w_ += self.eta * X.T.dot(errors) / X.shape[0]
            self.b_ += self.eta * errors.mean()
            loss = (-y.dot(np.log(output)) - (1 - y).dot(np.log(1 - output))) / X.shape[0]
            self.losses_.append(loss)
        return self

    def net_input(self, X):
        """Calculate net input"""
        return np.dot(X, self.w_) + self.b_

    def activation(self, z):
        """Compute logistic sigmoid activation"""
        return 1. / (1. + np.exp(-np.clip(z, -250, 250)))

    def predict(self, X):
        """Return class label after unit step"""
        return np.where(self.activation(self.net_input(X)) >= 0.5, 1, 0)

<br>
<br>

In [ ]:
X_train_01_subset = X_train_std[(y_train == 0) | (y_train == 1)]
y_train_01_subset = y_train[(y_train == 0) | (y_train == 1)]

lrgd = LogisticRegressionGD(eta=0.3, n_iter=1000, random_state=1)
lrgd.fit(X_train_01_subset,
         y_train_01_subset)

plot_decision_regions(X=X_train_01_subset, 
                      y=y_train_01_subset,
                      classifier=lrgd)

plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')

plt.tight_layout()
#plt.savefig('figures/03_05.png', dpi=300)
plt.show()

### Training a logistic regression model with scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=100.0, solver='lbfgs', multi_class='ovr')
lr.fit(X_train_std, y_train)

plot_decision_regions(X_combined_std, y_combined,
                      classifier=lr, test_idx=range(105, 150))
plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_06.png', dpi=300)
plt.show()

In [ ]:
lr.predict_proba(X_test_std[:3, :])

In [ ]:
lr.predict_proba(X_test_std[:3, :]).sum(axis=1)

In [ ]:
lr.predict_proba(X_test_std[:3, :]).argmax(axis=1)

In [ ]:
lr.predict(X_test_std[:3, :])

In [ ]:
lr.predict(X_test_std[0, :].reshape(1, -1))

<br>
<br>

### Tackling overfitting via regularization

In [ ]:
Image(filename='figures/03_07.png', width=700) 

In [ ]:
weights, params = [], []
for c in np.arange(-5, 5):
    lr = LogisticRegression(C=10.**c,
                            multi_class='ovr')
    lr.fit(X_train_std, y_train)
    weights.append(lr.coef_[1])
    params.append(10.**c)

weights = np.array(weights)
plt.plot(params, weights[:, 0],
         label='Petal length')
plt.plot(params, weights[:, 1], linestyle='--',
         label='Petal width')
plt.ylabel('Weight coefficient')
plt.xlabel('C')
plt.legend(loc='upper left')
plt.xscale('log')
#plt.savefig('figures/03_08.png', dpi=300)
plt.show()

<br>
<br>

# Maximum margin classification with support vector machines

In [ ]:
Image(filename='figures/03_09.png', width=700) 

## Maximum margin intuition

...

## Dealing with the nonlinearly separable case using slack variables

In [ ]:
Image(filename='figures/03_10.png', width=600) 

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='linear', C=1.0, random_state=1)
svm.fit(X_train_std, y_train)

plot_decision_regions(X_combined_std, 
                      y_combined,
                      classifier=svm, 
                      test_idx=range(105, 150))
plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_11.png', dpi=300)
plt.show()

## Alternative implementations in scikit-learn

In [ ]:
from sklearn.linear_model import SGDClassifier

ppn = SGDClassifier(loss='perceptron')
lr = SGDClassifier(loss='log')
svm = SGDClassifier(loss='hinge')

<br>
<br>

# Solving non-linear problems using a kernel SVM

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(1)
X_xor = np.random.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0,
                       X_xor[:, 1] > 0)
y_xor = np.where(y_xor, 1, 0)

plt.scatter(X_xor[y_xor == 1, 0],
            X_xor[y_xor == 1, 1],
            c='royalblue',
            marker='s',
            label='Class 1')
plt.scatter(X_xor[y_xor == 0, 0],
            X_xor[y_xor == 0, 1],
            c='tomato',
            marker='o',
            label='Class 0')

plt.xlim([-3, 3])
plt.ylim([-3, 3])
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.legend(loc='best')
plt.tight_layout()
#plt.savefig('figures/03_12.png', dpi=300)
plt.show()

In [ ]:
Image(filename='figures/03_13.png', width=700) 

<br>
<br>

## Using the kernel trick to find separating hyperplanes in higher dimensional space

In [ ]:
svm = SVC(kernel='rbf', random_state=1, gamma=0.10, C=10.0)
svm.fit(X_xor, y_xor)
plot_decision_regions(X_xor, y_xor,
                      classifier=svm)

plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_14.png', dpi=300)
plt.show()

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', random_state=1, gamma=0.2, C=1.0)
svm.fit(X_train_std, y_train)

plot_decision_regions(X_combined_std, y_combined,
                      classifier=svm, test_idx=range(105, 150))
plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_15.png', dpi=300)
plt.show()

In [ ]:
svm = SVC(kernel='rbf', random_state=1, gamma=100.0, C=1.0)
svm.fit(X_train_std, y_train)

plot_decision_regions(X_combined_std, y_combined, 
                      classifier=svm, test_idx=range(105, 150))
plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_16.png', dpi=300)
plt.show()

<br>
<br>

---

## **Decision tree learning**

Decision trees are highly **interpretable** models that classify data by asking a series of hierarchical questions. While often demonstrated with categorical data, they work equally well with continuous numerical features by applying "cut-off" values (e.g., "Is sepal width $\geq$ 2.8 cm?").

The process of decision trees are as follows:
* **Splitting:** The algorithm starts at the root and splits data based on the feature that provides the largest **Information Gain (IG)**.
* **Iteration:** This process repeats at each child node until the leaves are pure (containing only one class).
* **Regularization:** To prevent overfitting caused by excessively deep trees, practitioners typically **prune** the tree by setting a maximum depth limit.

<figure style='text-align: center'>
<img src='assets/03_17.png' style='width:500px'>
<figcaption>An  example of a decision tree</figcaption>
</figure>

### **Maximizing IG – getting the most bang for your buck**

To build an effective tree, the algorithm uses an objective function designed to maximize Information Gain (IG) at every split. Information Gain is the difference between the impurity of the parent node and the weighted sum of the impurities of its child nodes.

$$IG(D_p, f) = I(D_p) - \sum_{j=1}^m \frac{N_j}{N_p}I(D_j)$$

While the general formula allows for multiple branches, most standard libraries (like `scikit-learn`) implement binary decision trees, splitting a parent ($D_p$) into exactly two children ($D_{left}$ and $D_{right}$).

$$IG(D_p, f) = I(D_p) - \frac{N_{left}}{N_p}I(D_{left}) - \frac{N_{right}}{N_p}I(D_{right})$$


The Three Common Impurity Measures ($I$) are:

1. **Entropy ($I_H$):** Measures the uncertainty or "surprise." It is $0$ if a node is perfectly pure and maximal ($1.0$ in binary cases) if classes are distributed uniformly ($50/50$).
2. **Gini Impurity ($I_G$):** A measure of how often a randomly chosen element from the set would be incorrectly labeled.
3. **Classification Error ($I_E$):** Measures the misclassification rate.

#### **Shannon Entropy:**

The goal of using entropy as a criterion is to maximize mutual information.

$$I_H(t) = -\sum_{i=1}^c p(i \mid t) \log_2 p(i \mid t)$$

Here, $p(i \mid t)$ is the proportion of the examples that belong to class $i$ for a particular node, $t$. If $p=1$ or $p=0$, the node is "ordered" (Entropy = 0). If $p=0.5$, the node is at "maximum disorder" (Entropy = 1).

Let's visualize the entropy values for different class distributions via the following code:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def entropy(p):
    return - p * np.log2(p) - (1 - p) * np.log2((1 - p))

x = np.arange(0.0, 1.0, 0.01)
ent = [entropy(p) if p != 0 else None 
       for p in x]

plt.ylabel('Entropy')
plt.xlabel('Class-membership probability p(i=1)')
plt.plot(x, ent)
#plt.savefig('figures/03_26.png', dpi=300)
plt.show()
plt.close()

#### **Gini impurity**

Gini Impurity ($I_G$) is a criterion designed to minimize the probability of misclassification. It measures how often a randomly chosen element from a node would be incorrectly labeled if it were randomly labeled according to the distribution of labels in the subset.

$$I_G(t) = \sum_{i=1}^c p(i \mid t)(1 - p(i \mid t)) = 1 - \sum_{i=1}^c p(i|t)^2$$

Like entropy, it is maximal (0.5) when classes are perfectly mixed ($50/50$) and minimal (0) when a node is perfectly pure.

In the professional world, the choice between Gini and Entropy is rarely the "make or break" factor for a model. Because their curves are so similar in shape, they usually pick the same features to split on. Your time is much better spent **tuning hyperparameters** (like maximum depth or pruning) than debating which impurity measure to use. If you're looking for raw speed, Gini is often the default in libraries like `scikit-learn` because it avoids the computational cost of calculating logarithms.


#### **Classification error**

Classification Error ($I_E$) is the simplest impurity measure, representing the proportion of misclassified examples if the node were assigned the most frequent class label.

#$I_E(t) = 1 - \max{p(i \mid t)}$

It is primarily a useful criterion for pruning (trimming back a grown tree) but is not recommended for growing a tree. Because it is less sensitive to shifts in class probabilities than Gini or Entropy, it often fails to identify the splits that best "purify" the child nodes during the initial construction.

In the case of following two trees (A and B), the information gain using the classification error as a splitting criterion would be the same ($IG_E = 0.25$) in both scenarios, A and B. However, Gini impurity would favor the split in scenario B ($IG_G = 0.16$) over scenario A ($IG_G = 0.125$), which is indeed purer. Same with entropy criterion.

<figure style='text-align: center'>
<img src='assets/03_18.png' style='width:500px'>
<figcaption>Decision tree data split</figcaption>
</figure>

In [ ]:
def gini(p):
    return p * (1 - p) + (1 - p) * (1 - (1 - p))


def entropy(p):
    return - p * np.log2(p) - (1 - p) * np.log2((1 - p))


def error(p):
    return 1 - np.max([p, 1 - p])

x = np.arange(0.0, 1.0, 0.01)

ent = [entropy(p) if p != 0 else None for p in x]
sc_ent = [e * 0.5 if e else None for e in ent]
err = [error(i) for i in x]

fig = plt.figure()
ax = plt.subplot(111)
for i, lab, ls, c, in zip([ent, sc_ent, gini(x), err], 
                          ['Entropy', 'Entropy (scaled)', 
                           'Gini impurity', 'Misclassification error'],
                          ['-', '-', '--', '-.'],
                          ['black', 'lightgray', 'red', 'green', 'cyan']):
    line = ax.plot(x, i, label=lab, linestyle=ls, lw=2, color=c)

ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15),
          ncol=5, fancybox=True, shadow=False)

ax.axhline(y=0.5, linewidth=1, color='k', linestyle='--')
ax.axhline(y=1.0, linewidth=1, color='k', linestyle='--')
plt.ylim([0, 1.1])
plt.xlabel('p(i=1)')
plt.ylabel('Impurity index')
#plt.savefig('figures/03_19.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### **Building a decision tree**


In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
import numpy as np


# Load Iris dataset
iris = datasets.load_iris()

# Select last two features of the dataset
X = iris['data'][:, [2, 3]]
y = iris['target']
print('Class labels:', np.unique(y))

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from utils import plot_decision_regions


tree_model = DecisionTreeClassifier(criterion='gini', 
                                    max_depth=4, 
                                    random_state=1)
tree_model.fit(X_train, y_train)

X_combined = np.vstack((X_train, X_test))
y_combined = np.hstack((y_train, y_test))
plot_decision_regions(X_combined, y_combined, 
                      classifier=tree_model,
                      test_idx=range(105, 150))

plt.xlabel('Petal length [cm]')
plt.ylabel('Petal width [cm]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_20.png', dpi=300)
plt.show()
plt.close()

We can visualzie the decision tree above using the following code:

In [ ]:
from sklearn import tree

feature_names = ['Sepal length', 'Sepal width',
                 'Petal length', 'Petal width']
tree.plot_tree(tree_model,
               feature_names=feature_names,
               filled=True
               )

#plt.savefig('figures/03_21_1.pdf')
plt.show()
plt.close()

In [ ]:
tree_model_reg = DecisionTreeClassifier(max_depth=3)

tree_model_reg.fit(X_train, y_train)
X_combined = np.vstack((X_train, X_test))
y_combined = np.hstack((y_train, y_test))
plot_decision_regions(X_combined, y_combined, 
                      classifier=tree_model_reg,
                      test_idx=range(105, 150))

plt.xlabel('Petal length [cm]')
plt.ylabel('Petal width [cm]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_20.png', dpi=300)
plt.show()
plt.close()

In [ ]:
tree.plot_tree(
    tree_model_reg,
    feature_names=feature_names,
    filled=True,
    # max_depth=3
);

### **Combining weak to strong learners via random forests**

A Random Forest is an ensemble method that combines multiple deep decision trees to create a more robust model. Individual deep trees typically suffer from high variance (overfitting); by averaging their predictions, the random forest reduces variance, improves generalization, and increases robustness.

The 4-Step Random Forest Algorithm is as follows:

1. **Bootstrap Sampling:** Draw a random sample of size $n$ from the training dataset *with replacement*.
2. **Grow a Tree:** Build a decision tree from this bootstrap sample. At each node:
    * Randomly select $d$ features *without replacement*.
    * Split the node using the feature that maximizes the objective function (e.g., Information Gain).
3. **Repeat:** Repeat steps 1 and 2 $k$ times to create a forest of $k$ distinct trees.
4. **Aggregate:** Combine the predictions of all $k$ trees using a **majority vote** to assign the final class label.


#### **Hyperparamter tuning of Random Forests**

While Random Forests sacrifice the strict step-by-step interpretability of a single decision tree, they offer a major operational advantage: minimal hyperparameter tuning.

Unlike single trees, we typically do not need to prune individual trees in a forest. The ensemble structure naturally mitigates noise and overfitting through prediction averaging. The critical parameter to tune is the **number of trees ($k$)** in the forest. Increasing $k$ generally improves classification accuracy and stability, but comes at the cost of higher computational overhead during training and inference.

While Scikit-learn's default configurations usually suffice, we can fine-tune the bias-variance tradeoff of a Random Forest by adjusting the **bootstrap sample size ($n$)** and the **split feature subset size ($d$)**.

1. **Bootstrap Sample Size ($n$):** The size of the bootstrap sample directly controls the diversity of the individual trees:

    * **Smaller $n$ (High Randomness/Bias):** Decreasing sample size increases tree diversity because individual trees see less overlapping data. This reduces overfitting but can lower overall test performance (underfitting).
    * **Larger $n$ (Low Randomness/Variance):** Increasing sample size makes the bootstrap datasets more similar. The trees become correlated, which can cause the forest to overfit the original dataset.
    * **Industry Default:** Most implementations (like `scikit-learn`) set $n$ equal to the total number of training examples ($N$), which generally yields the optimal bias-variance balance.

2. **Feature Subset Size ($d$):** At each node split, the algorithm evaluates a random subset of $d$ features chosen from the total features ($m$).

    * **Standard Practice:** To ensure healthy randomization, $d$ should always be smaller than $m$.
    * **Industry Default:** The widely accepted and effective default is the square root of the total features:
    
    $$d = \sqrt{m}$$

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=25, 
                                random_state=1,
                                n_jobs=2)
forest.fit(X_train, y_train)

plot_decision_regions(X_combined, y_combined, 
                      classifier=forest, test_idx=range(105, 150))

plt.xlabel('Petal length [cm]')
plt.ylabel('Petal width [cm]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_2.png', dpi=300)
plt.show()
plt.close()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = forest.predict(X_test)

print(f"Training accuracy: {forest.score(X_train, y_train):0.2f}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):0.2f}")
print(classification_report(y_test, y_pred))

#### **Bonus:** How do bias/variance trade-off play out in  Random Forests

In Random Forests, when we average the predictions of multiple highly complex, unconstrained decision trees, we are exploiting statistical aggregation to systematically damp down variance.

Suppose we have an ensemble of $B$ identical, identically distributed (i.d.) trees, each with a variance of $\sigma^2$. If these trees were completely independent, the variance of their average would simply be $\frac{\sigma^2}{B}$, which approaches zero as $B \to \infty$.

* **Effect on variance:** Because the trees are trained on the same underlying dataset, they are correlated. If the positive pairwise correlation between any two trees is $\rho$, the exact analytical variance of the ensemble average is:

    $$\text{Var}\left(\frac{1}{B}\sum_{i=1}^{B} X_i\right) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$$

    So, as $B$ (the number of trees) increases, the second term $\frac{1-\rho}{B}\sigma^2$ shrinks toward zero and the variance of the ensemble plateaus at $\rho \sigma^2$. The takeaway is that the final variance is strictly bound by $\rho$ (how correlated the trees are) and averaging drastically reduces variance compared to an individual tree, provided the trees aren't perfectly correlated ($\rho < 1$).

* **Effect on bias:** Because the final prediction is a simple linear combination (an average) of the individual expected values, the bias of the ensemble is identical to the bias of a single individual tree. In practice, because bootstrap sampling (in RF) slightly limits the data choices available to any single tree, individual trees cannot fit the training data *quite* as perfectly as a single standard decision tree. Therefore, individual tree bias increases slightly, but this minor sacrifice is heavily outweighed by the massive reduction in variance.

#### **Bonus:** What is extremely randomized trees (or Extra Trees)?

Extremely Randomized Trees (commonly referred to as Extra Trees or ERF) is a tree-based ensemble learning algorithm designed by Pierre Geurts, Damien Ernst, and Louis Wehenkel in 2006. From a first-principles perspective, Extra Trees is a variant of Random Forests built to push the Bias-Variance Trade-off to an extreme. It purposefully sacrifices individual tree accuracy (increasing individual bias) to achieve a massive reduction in the overall ensemble variance.

To understand Extra Trees, it is best to compare its core mechanics directly against a standard Random Forest. Both algorithms build an ensemble of independent decision trees and average their predictions, but Extra Trees introduces a different randomization strategy.

##### **Level 1 randomization: Data sampling**

* **Random Forest:** Uses Bootstrap Aggregation (Bagging). Each tree is trained on a unique, randomly sampled subset of rows (roughly 63.2% of the original dataset), which naturally de-correlates the trees.
* **Extra Trees:** By default, each tree uses the entire, raw training dataset (`bootstrap=False`). It does not rely on row-level sampling to create diversity.

##### **Level 2:randomization Node splitting mechanics**

* **Random Forest:** For each of those $m$ features, the algorithm mathematically scans through *every single possible split point* in the data to find the exact threshold that maximizes information gain (e.g., minimizing Gini Impurity or Mean Squared Error). This is a greedy, locally optimized split.
* **Extra Trees:** Instead of hunting for the best possible threshold, the algorithm draws a completely random split point (uniformly distributed between the feature's minimum and maximum values) for each of the $m$ features. It then evaluates those $m$ random thresholds and picks the best one among them.


##### **Bias/variance trade-off in ERF models**

As we describe in the previous bonus topic, the variance of an ensemble average of $B$ correlated trees can be calculated using the following equation:

$$\text{Var}(\text{Ensemble}) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$$

Where $\sigma^2$ is the variance of an individual tree and $\rho$ is the correlation between any two trees.

Because Random Forests look for the absolute optimal mathematical boundaries, trees will naturally choose similar split thresholds if they share features. By choosing thresholds completely at random, Extra Trees forces the structures of the trees to be radically different from one another, driving $\rho$ close to zero.

Additionally, random split thresholds act as a heavy structural regularizer. Individual trees are prevented from perfectly micro-carving the feature space to fit highly localized noise.

On the bias side, however, the outcome is a little different. Because the split points are randomized rather than optimized, individual trees are structurally weaker and have higher bias than Random Forest trees. However, when we average hundreds of these heavily de-correlated trees together, the reduction in variance completely overrides the slight bias increase, leading to excellent generalization performance.

##### **Practical engineering advantages**

* **Massive speed up in training latency:** The most computationally expensive step in tree-based modeling is sorting the data at every single node to find the optimal mathematical split point. Because Extra Trees bypasses this search entirely by picking a random number, Extra Trees trains significantly faster than Random Forests or Gradient Boosted Trees, especially on datasets with high numbers of continuous features.

* **Robustness to overfitting on continuous noise:** If our dataset contains features with smooth, noisy continuous variables, a standard Random Forest will meticulously fit splits directly to those tiny variations. Extra Trees' random thresholds smoothly slice through the noise, making the model highly robust to variance in continuous data.

##### **When to use ERF in production**

* **Use Extra Trees if:** You are working with high-dimensional tabular data, your compute/time budget for training is highly constrained, and your baseline Random Forest is severely overfitting (exhibiting high variance).
* **Avoid Extra Trees if:** Your features are mostly sparse, low-cardinality categorical encodings (where drawing a random threshold uniformly doesn't provide meaningful architectural diversity) or if your dataset is so small that the algorithm's high bias tax prevents it from learning the true underlying signal.

In scikit-learn, we can easily deploy this architecture using `ExtraTreesClassifier` or `ExtraTreesRegressor`.

---

# K-nearest neighbors - a lazy learning algorithm

In [ ]:
Image(filename='figures/03_23.png', width=400) 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, 
                           p=2, 
                           metric='minkowski')
knn.fit(X_train_std, y_train)

plot_decision_regions(X_combined_std, y_combined, 
                      classifier=knn, test_idx=range(105, 150))

plt.xlabel('Petal length [standardized]')
plt.ylabel('Petal width [standardized]')
plt.legend(loc='upper left')
plt.tight_layout()
#plt.savefig('figures/03_24_figures.png', dpi=300)
plt.show()

<br>
<br>

# Summary

...

---

Readers may ignore the next cell.

In [ ]:
! python ../.convert_notebook_to_script.py --input ch03.ipynb --output ch03.py